<a href="https://colab.research.google.com/github/javidsan/PROGCOM-B/blob/main/app_financiera_progcom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Asesor financiero · PY
import calendar
import csv
import getpass
import hashlib
import json
import os
import uuid
from datetime import date, datetime, timedelta

CARPETA = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
ARCHIVO = os.path.join(CARPETA, "asesor_datos.json")
CATS_INICIALES = ["Comida", "Transporte", "Vivienda", "Servicios",
                  "Salud", "Ocio", "Deudas", "Otros"]
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
         "agosto", "septiembre", "octubre", "noviembre", "diciembre"]


def completar(d):
    d.setdefault("cats", list(CATS_INICIALES))
    for clave in ("tx", "debts", "owed", "del"):
        d.setdefault(clave, [])
    return d


def cargar():
    if os.path.exists(ARCHIVO):
        with open(ARCHIVO, encoding="utf-8") as f:
            return completar(json.load(f))
    return completar({})


def guardar():
    with open(ARCHIVO, "w", encoding="utf-8") as f:
        json.dump(S, f, ensure_ascii=False, indent=1)


S = cargar()


def plata(n):
    texto = f"{round(abs(n)):,}".replace(",", ".")
    return ("-" if n < 0 else "") + "$" + texto


def nuevo_id():
    return uuid.uuid4().hex[:8]


def hoy():
    return date.today().isoformat()


def dias_para(fecha):
    return (date.fromisoformat(fecha) - date.today()).days


def siguiente_mes(fecha):
    d = date.fromisoformat(fecha)
    mes = d.month % 12 + 1
    anio = d.year + (1 if d.month == 12 else 0)
    dia = min(d.day, calendar.monthrange(anio, mes)[1])
    return date(anio, mes, dia).isoformat()


def nombre_mes(m):
    return MESES[int(m[5:7]) - 1] + " " + m[:4]


def pedir_numero(texto, obligatorio=True):
    while True:
        r = input(texto).strip().replace(",", ".")
        if r == "" and not obligatorio:
            return 0.0
        try:
            n = float(r)
            if n > 0 or (n == 0 and not obligatorio):
                return int(n) if n == int(n) else n
        except ValueError:
            pass
        print("  -> Escribe un número mayor a 0.")


def pedir_fecha(texto, opcional=False, por_defecto=None):
    while True:
        r = input(texto).strip()
        if r == "":
            if por_defecto:
                return por_defecto
            if opcional:
                return ""
        try:
            return date.fromisoformat(r).isoformat()
        except ValueError:
            print("  -> Usa el formato AAAA-MM-DD (ej. 2026-12-31).")


def si_no(texto):
    return input(texto + " (s/n): ").strip().lower() == "s"


def menu(titulo, opciones):
    print(f"\n=== {titulo} ===")
    for i, o in enumerate(opciones, 1):
        print(f"  {i}. {o}")
    print("  0. Volver")
    r = input("Elige: ").strip()
    if r.isdigit() and int(r) <= len(opciones):
        return int(r)
    print("  -> Opción no válida.")
    return -1


def hacer_hash(texto):
    return hashlib.sha256(("afin:" + texto).encode()).hexdigest()


def crear_contrasena():
    while True:
        a = getpass.getpass("Nueva contraseña (mínimo 4 caracteres): ")
        if len(a) < 4:
            print("  -> Muy corta.")
            continue
        if a != getpass.getpass("Repítela: "):
            print("  -> No coinciden.")
            continue
        S["ph"] = hacer_hash(a)
        S["nolock"] = False
        guardar()
        print("Contraseña guardada.")
        return True


def pedir_contrasena():
    if hacer_hash(getpass.getpass("Contraseña: ")) == S.get("ph"):
        return True
    print("  -> Contraseña incorrecta.")
    return False


def entrar():
    if not S.get("ph") or S.get("nolock"):
        return True
    for _ in range(3):
        if pedir_contrasena():
            return True
    return False


def confirmar_borrado(tipo, nombre, monto, fecha):
    if not S.get("ph"):
        print("Para borrar cosas primero necesitas una contraseña.")
        crear_contrasena()
    print(f"¿Seguro que quieres borrar «{nombre}»? Quedará en Borradas.")
    motivo = input("Motivo (obligatorio, Enter para cancelar): ").strip()
    if not motivo or not pedir_contrasena():
        print("Cancelado.")
        return False
    S["del"].append({"id": nuevo_id(), "t": tipo, "n": nombre, "a": monto,
                     "f": fecha, "r": motivo,
                     "w": datetime.now().strftime("%Y-%m-%d %H:%M")})
    guardar()
    print("Borrado. Quedó anotado en Borradas.")
    return True


def en_periodo(fecha, p):
    h = date.today()
    if p == "d":
        return fecha == h.isoformat()
    if p == "w":
        lunes = (h - timedelta(days=h.weekday())).isoformat()
        return lunes <= fecha <= h.isoformat()
    return fecha.startswith(h.isoformat()[:7])


def totales(p="m"):
    ing = sum(t["a"] for t in S["tx"] if t["k"] == "i" and en_periodo(t["d"], p))
    gas = sum(t["a"] for t in S["tx"] if t["k"] == "g" and en_periodo(t["d"], p))
    return ing, gas, ing - gas


def deudas_activas():
    return [d for d in S["debts"] if d["amount"] > 0]


def cobros_activos():
    return [o for o in S["owed"] if o["amount"] > 0]


def texto_vence(fecha):
    n = dias_para(fecha)
    if n < 0:
        return f"vencida hace {-n} días"
    if n == 0:
        return "vence hoy"
    if n == 1:
        return "vence mañana"
    return f"vence en {n} días"


def resumen_meses():
    datos = {}
    for t in S["tx"]:
        m = t["d"][:7]
        datos.setdefault(m, {"i": 0, "e": 0})
        datos[m]["i" if t["k"] == "i" else "e"] += t["a"]
    actual = hoy()[:7]
    inicio = min(list(datos) + [actual])
    lista = []
    y, mo = int(inicio[:4]), int(inicio[5:7])
    while True:
        m = f"{y}-{mo:02d}"
        v = datos.get(m, {"i": 0, "e": 0})
        lista.append({"m": m, "i": v["i"], "e": v["e"], "r": v["i"] - v["e"],
                      "abierto": m == actual})
        if m >= actual:
            return lista
        mo += 1
        if mo > 12:
            mo, y = 1, y + 1


def porcentaje(resultado, ingresos):
    return None if ingresos <= 0 else resultado / ingresos * 100


def mostrar_alertas():
    for d in sorted(deudas_activas(), key=lambda x: x["due"]):
        if dias_para(d["due"]) <= 3:
            print(f"  [ALERTA] {d['name']}: {texto_vence(d['due'])} · {d['due']} "
                  f"({plata(d.get('min') or d['amount'])})")
    for o in cobros_activos():
        if o.get("due") and dias_para(o["due"]) < 0:
            print(f"  [ALERTA] {o['name']} te debía pagar {plata(o['amount'])} "
                  f"hace {-dias_para(o['due'])} días")
    cerrados = [x for x in resumen_meses()
                if not x["abierto"] and (x["i"] or x["e"]) and x["m"] > S.get("ack", "")]
    if cerrados:
        x = cerrados[-1]
        print(f"  [INFO] Se cerró {nombre_mes(x['m'])}: "
              f"{'pérdida' if x['r'] < 0 else 'ganancia'} de {plata(abs(x['r']))}.")
        if si_no("  ¿Ocultar este aviso?"):
            S["ack"] = x["m"]
            guardar()


def pantalla_inicio():
    p = S.get("per", "m")
    nombres = {"d": "hoy", "w": "esta semana", "m": "este mes"}
    ing, gas, libre = totales(p)
    deudas = sum(d["amount"] for d in deudas_activas())
    cobrar = sum(o["amount"] for o in cobros_activos())
    print("\n" + "=" * 40)
    if ing or gas:
        print(f"Te sobran {nombres[p]}:  {plata(libre)}")
        if libre < 0:
            print("Estás gastando más de lo que entra. Mira el Asesor.")
    else:
        print("Empieza registrando tus ingresos y gastos.")
    print("=" * 40)
    mostrar_alertas()
    print(f"\nIngresos: {plata(ing)}   Gastos: {plata(gas)}")
    print(f"Deudas pendientes: {plata(deudas)}   Te deben: {plata(cobrar)}")


    gasto = {}
    for t in S["tx"]:
        if t["k"] == "g" and en_periodo(t["d"], p):
            gasto[t["c"]] = gasto.get(t["c"], 0) + t["a"]
    mayor = max([1] + list(gasto.values()))
    print(f"\nMis apartados ({nombres[p]}):")
    for c in S["cats"]:
        v = gasto.get(c, 0)
        print(f"  {c:<14} {'#' * int(v / mayor * 20):<20} {plata(v)}")


def menu_inicio():
    while True:
        pantalla_inicio()
        op = menu("Inicio", ["Cambiar periodo (hoy/semana/mes)",
                             "Crear apartado", "Eliminar apartado"])
        if op == 0:
            return
        if op == 1:
            r = input("¿Qué periodo? d = hoy, w = semana, m = mes: ").strip().lower()
            if r in ("d", "w", "m"):
                S["per"] = r
                guardar()
        elif op == 2:
            nombre = input("Nombre del apartado nuevo: ").strip()
            if not nombre:
                print("  -> Escribe un nombre.")
            elif nombre in S["cats"]:
                print("  -> Ya existe ese apartado.")
            else:
                S["cats"].append(nombre)
                guardar()
        elif op == 3:
            for i, c in enumerate(S["cats"], 1):
                print(f"  {i}. {c}")
            r = input("Número del apartado a eliminar (Enter = cancelar): ").strip()
            if r.isdigit() and 1 <= int(r) <= len(S["cats"]):
                if not S.get("ph"):
                    print("Primero crea una contraseña.")
                    crear_contrasena()
                if pedir_contrasena():
                    quitado = S["cats"].pop(int(r) - 1)
                    guardar()
                    print(f"Apartado «{quitado}» eliminado (sus movimientos no se borran).")


def anotar_movimiento():
    k = input("Tipo: g = gasto, i = ingreso: ").strip().lower()
    if k not in ("g", "i"):
        print("  -> Elige g o i.")
        return
    monto = pedir_numero("Monto: ")
    categoria = "Ingreso"
    if k == "g":
        for i, c in enumerate(S["cats"], 1):
            print(f"  {i}. {c}")
        r = input("Número del apartado: ").strip()
        if not (r.isdigit() and 1 <= int(r) <= len(S["cats"])):
            print("  -> Apartado no válido.")
            return
        categoria = S["cats"][int(r) - 1]
    nota = input("Nota (opcional): ").strip()
    fecha = pedir_fecha("Fecha AAAA-MM-DD (Enter = hoy): ", por_defecto=hoy())
    S["tx"].append({"id": nuevo_id(), "k": k, "a": monto, "c": categoria,
                    "n": nota, "d": fecha})
    guardar()
    print("Guardado.")


def ver_movimientos():
    limite = 5
    while True:
        todos = sorted(reversed(S["tx"]), key=lambda t: t["d"], reverse=True)
        if not todos:
            print("\nAún no hay movimientos.")
            return
        print("\n--- Últimos movimientos ---")
        for i, t in enumerate(todos[:limite], 1):
            signo = "+" if t["k"] == "i" else "-"
            print(f"{i:>3}. {t['d']}  {signo}{plata(t['a']):<12} {t['c']:<12} {t.get('n', '')}")
        r = input("Número para borrar, 'm' para ver más, Enter para volver: ").strip().lower()
        if r == "":
            return
        if r == "m":
            limite += 5
        elif r.isdigit() and 1 <= int(r) <= min(limite, len(todos)):
            t = todos[int(r) - 1]
            if confirmar_borrado("Movimiento", t.get("n") or t["c"], t["a"], t["d"]):
                S["tx"].remove(t)
                guardar()


def ver_borradas():
    print("\n--- Transacciones borradas ---")
    if not S["del"]:
        print("Aún no has borrado nada.")
    for x in reversed(S["del"]):
        print(f"{x['t']} · {x['n']} · {plata(x['a'])}"
              f"{' · fecha ' + x['f'] if x.get('f') else ''}")
        print(f"   Motivo: {x['r']}   (borrado el {x['w']})")


def csv_mes(m):
    x = next(y for y in resumen_meses() if y["m"] == m)
    archivo = os.path.join(CARPETA, f"cierre-{m}.csv")
    p = porcentaje(x["r"], x["i"])
    with open(archivo, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f, delimiter=";", quoting=csv.QUOTE_ALL)
        w.writerow(["Cierre de " + nombre_mes(m)])
        w.writerow(["Ingresos", x["i"]])
        w.writerow(["Gastos", x["e"]])
        w.writerow(["Pérdida" if x["r"] < 0 else "Ganancia", abs(x["r"])])
        w.writerow(["Porcentaje", "sin ingresos" if p is None else f"{abs(p):.1f}%"])
        w.writerow([])
        w.writerow(["Fecha", "Tipo", "Apartado", "Nota", "Monto"])
        for t in sorted((t for t in S["tx"] if t["d"].startswith(m)), key=lambda t: t["d"]):
            w.writerow([t["d"], "Ingreso" if t["k"] == "i" else "Gasto",
                        t["c"], t.get("n", ""), t["a"]])
    print(f"Archivo creado: {archivo}  (ábrelo en Excel o Google Sheets)")


def menu_cierres():
    lista = list(reversed(resumen_meses()))
    print("\n--- Cierres mensuales ---")
    for i, x in enumerate(lista, 1):
        p = porcentaje(x["r"], x["i"])
        pct = "" if p is None else f" {abs(p):.1f}%"
        print(f"{i:>3}. {nombre_mes(x['m']):<16} [{'en curso' if x['abierto'] else 'cerrado'}] "
              f"Ingresos {plata(x['i'])} · Gastos {plata(x['e'])} · "
              f"{'Pérdida' if x['r'] < 0 else 'Ganancia'} {plata(abs(x['r']))}{pct}")
    r = input("Números a comparar separados por coma (ej. 1,2), "
              "'c' + número para descargar CSV (ej. c1), Enter = volver: ").strip().lower()
    if r.startswith("c") and r[1:].isdigit() and 1 <= int(r[1:]) <= len(lista):
        csv_mes(lista[int(r[1:]) - 1]["m"])
    elif r:
        try:
            elegidos = [lista[int(n) - 1] for n in r.split(",")]
        except (ValueError, IndexError):
            print("  -> Números no válidos.")
            return
        ing = sum(x["i"] for x in elegidos)
        res = sum(x["r"] for x in elegidos)
        p = porcentaje(res, ing)
        print(f"\n{len(elegidos)} mes(es): Ingresos {plata(ing)} · "
              f"{'Pérdida' if res < 0 else 'Ganancia'} {plata(abs(res))}"
              f"{'' if p is None else f' ({abs(p):.1f}% de lo que ingresó)'}")
        if len(elegidos) == 2 and elegidos[0]["r"] != 0:
            b, a = sorted(elegidos, key=lambda x: x["m"])
            cambio = (a["r"] - b["r"]) / abs(b["r"]) * 100
            print(f"De {nombre_mes(b['m'])} a {nombre_mes(a['m'])}, tu resultado "
                  f"{'mejoró' if a['r'] >= b['r'] else 'empeoró'} {abs(cambio):.1f}%")


def menu_movimientos():
    while True:
        op = menu("Movimientos", ["Anotar movimiento", "Ver / borrar movimientos",
                                  "Cierres mensuales", "Borradas"])
        if op == 0:
            return
        if op == 1:
            anotar_movimiento()
        elif op == 2:
            ver_movimientos()
        elif op == 3:
            menu_cierres()
        elif op == 4:
            ver_borradas()


def listar_deudas():
    lista = sorted(S["debts"], key=lambda d: (d["amount"] <= 0, d["due"]))
    print("\n--- Mis deudas ---")
    if not lista:
        print("No tienes deudas registradas.")
    for i, d in enumerate(lista, 1):
        if d["amount"] > 0:
            print(f"{i:>3}. {d['name']}: falta {plata(d['amount'])} · cuota {plata(d.get('min', 0))} "
                  f"· interés {d.get('rate', 0)}% · pago {d['due']} ({texto_vence(d['due'])})")
        else:
            print(f"{i:>3}. {d['name']}: PAGADA el {d.get('paid', '(sin fecha)')}")
        for p in d.get("pays", []):
            print(f"       Pagó {plata(p['a'])} el {p['d']}")
    return lista


def agregar_deuda():
    nombre = input("¿A quién le debes?: ").strip()
    if not nombre:
        print("  -> Escribe un nombre.")
        return
    S["debts"].append({
        "id": nuevo_id(), "name": nombre,
        "amount": pedir_numero("Cuánto falta por pagar: "),
        "min": pedir_numero("Pago mensual (Enter = 0): ", obligatorio=False),
        "rate": pedir_numero("Interés mensual en % (Enter = 0): ", obligatorio=False),
        "due": pedir_fecha("Próxima fecha de pago AAAA-MM-DD: "),
        "alarm": True})
    guardar()
    print("Deuda guardada.")


def pagar_deuda(d):
    a = pedir_numero(f"Monto pagado (Enter = {d.get('min') or d['amount']}): ",
                     obligatorio=False) or (d.get("min") or d["amount"])
    pago = min(a, d["amount"])
    d["amount"] -= pago
    d.setdefault("pays", []).append({"d": hoy(), "a": pago})
    S["tx"].append({"id": nuevo_id(), "k": "g", "a": pago, "c": "Deudas",
                    "n": "Pago: " + d["name"], "d": hoy()})
    if d["amount"] <= 0:
        d["paid"] = hoy()
        print("¡Deuda pagada!")
    else:
        d["due"] = siguiente_mes(d["due"])
        print(f"Pago registrado. Falta {plata(d['amount'])}. Próxima fecha: {d['due']}")
    guardar()


def menu_deudas():
    while True:
        lista = listar_deudas()
        op = menu("Deudas", ["Agregar deuda", "Registrar pago", "Cambiar nombre", "Eliminar"])
        if op == 0:
            return
        if op == 1:
            agregar_deuda()
            continue
        if op in (2, 3, 4):
            r = input("Número de la deuda: ").strip()
            if not (r.isdigit() and 1 <= int(r) <= len(lista)):
                print("  -> Número no válido.")
                continue
            d = lista[int(r) - 1]
            if op == 2:
                if d["amount"] > 0:
                    pagar_deuda(d)
                else:
                    print("  -> Esa deuda ya está pagada.")
            elif op == 3:
                nuevo = input("Nombre nuevo: ").strip()
                if nuevo:
                    for t in S["tx"]:
                        if t["c"] == "Deudas" and t.get("n") == "Pago: " + d["name"]:
                            t["n"] = "Pago: " + nuevo
                    d["name"] = nuevo
                    guardar()
            elif op == 4:
                if confirmar_borrado("Deuda", d["name"], d["amount"], d["due"]):
                    S["debts"].remove(d)
                    guardar()


def suma_abonos(o, regalo):
    return sum(x["a"] for x in o.get("ab", []) if bool(x.get("g")) == regalo)


def listar_cobros():
    lista = sorted(S["owed"], key=lambda o: (o["amount"] <= 0, o.get("due") or "9999"))
    print(f"\n--- Me deben: {plata(sum(o['amount'] for o in cobros_activos()))} ---")
    if not lista:
        print("Nadie te debe por ahora.")
    for i, o in enumerate(lista, 1):
        total = o.get("total") or o["amount"]
        estado = "SALDADA" if o["amount"] <= 0 else (
            texto_vence(o["due"]) if o.get("due") else "sin fecha")
        print(f"{i:>3}. {o['name']}{' (' + o['note'] + ')' if o.get('note') else ''}: "
              f"debía {plata(total)} · abonó {plata(suma_abonos(o, False))}"
              f"{' · obsequió ' + plata(suma_abonos(o, True)) if suma_abonos(o, True) else ''}"
              f" · falta {plata(o['amount'])} [{estado}]")
        for x in o.get("ab", []):
            print(f"       {'Obsequió' if x.get('g') else 'Abonó'} {plata(x['a'])} el {x['d']}")
    return lista


def agregar_cobro():
    nombre = input("¿Quién te debe?: ").strip()
    if not nombre:
        print("  -> Escribe un nombre.")
        return
    monto = pedir_numero("Cuánto te debe: ")
    nota = input("Por qué te debe (opcional): ").strip()
    fecha = pedir_fecha("Fecha en que prometió pagar AAAA-MM-DD (Enter = sin fecha): ", opcional=True)
    S["owed"].append({"id": nuevo_id(), "name": nombre, "amount": monto,
                      "total": monto, "ab": [], "note": nota, "due": fecha})
    if si_no("¿Descontar de lo que tienes (lo presté hoy)?"):
        if "Préstamos" not in S["cats"]:
            S["cats"].append("Préstamos")
        S["tx"].append({"id": nuevo_id(), "k": "g", "a": monto, "c": "Préstamos",
                        "n": "Préstamo a " + nombre, "d": hoy()})
    guardar()
    print("Guardado.")


def abonar_cobro(o, regalo):
    a = pedir_numero("Monto a obsequiar: " if regalo else "Cuánto abonó: ")
    pago = min(a, o["amount"])
    o.setdefault("total", o["amount"])
    o.setdefault("ab", []).append({"d": hoy(), "a": pago, **({"g": True} if regalo else {})})
    o["amount"] -= pago
    if not regalo:
        S["tx"].append({"id": nuevo_id(), "k": "i", "a": pago, "c": "Ingreso",
                        "n": "Abono de " + o["name"], "d": hoy()})
    guardar()
    print("Saldada." if o["amount"] <= 0 else f"Registrado. Falta {plata(o['amount'])}.")


def menu_cobros():
    while True:
        lista = listar_cobros()
        op = menu("Me deben", ["Agregar", "Registrar abono", "Obsequiar", "Eliminar"])
        if op == 0:
            return
        if op == 1:
            agregar_cobro()
            continue
        if op in (2, 3, 4):
            r = input("Número: ").strip()
            if not (r.isdigit() and 1 <= int(r) <= len(lista)):
                print("  -> Número no válido.")
                continue
            o = lista[int(r) - 1]
            if op in (2, 3):
                if o["amount"] > 0:
                    abonar_cobro(o, regalo=(op == 3))
                else:
                    print("  -> Ya está saldada.")
            elif confirmar_borrado("Me deben", o["name"], o["amount"], o.get("due", "")):
                S["owed"].remove(o)
                guardar()


def plan_deudas():
    ds = deudas_activas()
    if not ds:
        print("\nNo tienes deudas pendientes. Si las agregas en Deudas, te armo un plan.")
        return
    print("\n1. Pagar menos intereses (la que más interés cobra primero)")
    print("2. Salir rápido de deudas (la más pequeña primero)")
    avalancha = input("Elige 1 o 2: ").strip() != "2"
    ds.sort(key=(lambda d: -d.get("rate", 0)) if avalancha else (lambda d: d["amount"]))
    ing, gas, libre = totales()
    intereses = sum(d["amount"] * d.get("rate", 0) / 100 for d in ds)
    print(f"\nIntereses que pagas al mes: {plata(intereses)}")
    print("Paga la cuota normal en todas y manda lo extra a la primera de la lista:")
    for i, d in enumerate(ds, 1):
        extra = f"  <- aquí va lo extra{f' ({plata(libre)} que te sobran)' if libre > 0 else ''}" if i == 1 else ""
        print(f"  {i}. {d['name']} · falta {plata(d['amount'])} · {d.get('rate', 0)}% mensual{extra}")
    print("\nIdeas que funcionan:")
    print(" - Nunca pagues solo el mínimo si puedes más: el interés se come tu abono.")
    print(" - Pregunta si puedes bajar la tasa o unir deudas en una más barata.")
    print(" - Cada gasto que recortes, mándalo directo a la deuda.")
    print(" - Evita deudas nuevas mientras pagas las actuales.")
    if cobros_activos():
        print(" - Si te deben plata, cobrarla y usarla para abonar te ahorra intereses.")


def aconsejar_compra():
    nombre = input("¿Qué quieres comprar?: ").strip() or "esta compra"
    precio = pedir_numero("Precio: ")
    necesidad = input("¿Necesidad (n) o gusto (g)?: ").strip().lower() != "g"
    cuotas = input("¿Contado (c) o cuotas/crédito (q)?: ").strip().lower() == "q"

    ing, gas, libre = totales()
    nivel = 0
    razones = []
    vencidas = any(dias_para(d["due"]) < 0 for d in deudas_activas())
    con_interes = any(d.get("rate", 0) > 0 for d in deudas_activas())

    if ing == 0:
        razones.append("Este mes no has registrado ingresos, así que no puedo medir cuánto te sobra.")
        nivel = max(nivel, 1)
    elif precio > libre:
        razones.append(f"Cuesta más de lo que te sobra este mes ({plata(libre)}).")
        nivel = max(nivel, 1 if necesidad else 2)
        if necesidad:
            razones.append("Como es una necesidad, busca una opción más barata o ahorra por partes.")
    elif precio > libre / 2:
        razones.append("Se llevaría más de la mitad de lo que te sobra este mes.")
        nivel = max(nivel, 1)

    if not necesidad and vencidas:
        razones.append("Tienes deudas vencidas. Ponte al día antes de comprar un gusto.")
        nivel = 2
    elif not necesidad and con_interes:
        razones.append("Tienes deudas con interés. Abonarles te ahorra más que esta compra.")
        nivel = max(nivel, 1)

    por_cobrar = sum(o["amount"] for o in cobros_activos())
    if por_cobrar > 0 and precio > libre:
        razones.append(f"Te deben {plata(por_cobrar)}, pero no cuentes con ese dinero hasta que lo recibas.")
    if cuotas:
        razones.append("A cuotas casi siempre terminas pagando más. Si puedes, ahorra y paga de contado.")
        nivel = max(nivel, 1)
    if not necesidad:
        razones.append("Espera 24 horas. Si mañana lo sigues queriendo, decide con calma.")
    if not razones or (nivel == 0 and necesidad):
        razones.insert(0, "Cabe en tu presupuesto de este mes.")

    veredicto = ["Puedes comprarlo", "Piénsalo un poco más", "Mejor no por ahora"][nivel]
    print(f"\n>>> {veredicto}: {nombre} ({plata(precio)})")
    for r in razones:
        print(f" - {r}")


def menu_asesor():
    while True:
        op = menu("Asesor", ["Plan para reducir deudas", "Voy a hacer una compra"])
        if op == 0:
            return
        if op == 1:
            plan_deudas()
        elif op == 2:
            aconsejar_compra()


def guardar_copia():
    archivo = os.path.join(CARPETA, f"asesor-financiero-{hoy()}.json")
    with open(archivo, "w", encoding="utf-8") as f:
        json.dump(S, f, ensure_ascii=False)
    print(f"Copia guardada en: {archivo}")


def cargar_copia():
    global S
    ruta = input("Ruta del archivo .json (puedes arrastrarlo aquí): ").strip().strip('"')
    try:
        with open(ruta, encoding="utf-8") as f:
            nuevo = json.load(f)
        if not isinstance(nuevo.get("tx"), list) or not isinstance(nuevo.get("cats"), list):
            raise ValueError
    except (OSError, ValueError):
        print("  -> El archivo no es una copia válida.")
        return
    print(f"Tiene {len(nuevo['tx'])} movimientos y {len(nuevo.get('debts', []))} deudas.")
    if si_no("Esto REEMPLAZA todo lo que hay aquí. ¿Cargar?"):
        S = completar(nuevo)
        guardar()
        print("Copia cargada.")


def menu_datos():
    global S
    while True:
        pide = bool(S.get("ph")) and not S.get("nolock")
        op = menu("Mis datos", ["Guardar copia en archivo", "Cargar copia de archivo",
                                "Crear / cambiar contraseña",
                                f"Contraseña al abrir: {'Sí' if pide else 'No'} (cambiar)"])
        if op == 0:
            return
        if op == 1:
            guardar_copia()
        elif op == 2:
            cargar_copia()
        elif op == 3:
            if not S.get("ph") or pedir_contrasena():
                crear_contrasena()
        elif op == 4:
            if not S.get("ph"):
                crear_contrasena()
            elif S.get("nolock"):
                S["nolock"] = False
                guardar()
                print("Ahora pedirá contraseña al abrir.")
            elif pedir_contrasena():
                S["nolock"] = True
                guardar()
                print("Ya no pedirá contraseña al abrir.")


def main():
    if not entrar():
        print("No se pudo entrar. Chao.")
        return
    print("\nMI ASESOR FINANCIERO")
    mostrar_alertas()
    while True:
        op = menu("Menú principal", ["Inicio", "Movimientos", "Deudas",
                                     "Me deben", "Asesor", "Mis datos"])
        if op == 0:
            print("¡Hasta pronto!")
            return
        if op == 1:
            menu_inicio()
        elif op == 2:
            menu_movimientos()
        elif op == 3:
            menu_deudas()
        elif op == 4:
            menu_cobros()
        elif op == 5:
            menu_asesor()
        elif op == 6:
            menu_datos()


if __name__ == "__main__":
    try:
        main()
    except (KeyboardInterrupt, EOFError):
        print("\nCerrado. Tus datos ya estaban guardados.")


MI ASESOR FINANCIERO

=== Menú principal ===
  1. Inicio
  2. Movimientos
  3. Deudas
  4. Me deben
  5. Asesor
  6. Mis datos
  0. Volver
Elige: 2

=== Movimientos ===
  1. Anotar movimiento
  2. Ver / borrar movimientos
  3. Cierres mensuales
  4. Borradas
  0. Volver
Elige: 4

--- Transacciones borradas ---
Aún no has borrado nada.

=== Movimientos ===
  1. Anotar movimiento
  2. Ver / borrar movimientos
  3. Cierres mensuales
  4. Borradas
  0. Volver
Elige: 0

=== Menú principal ===
  1. Inicio
  2. Movimientos
  3. Deudas
  4. Me deben
  5. Asesor
  6. Mis datos
  0. Volver
